In [1]:
from __future__ import print_function
import numpy as np
from numpy import newaxis as na
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import os
import sys
import numpy as np
import shap
import warnings
warnings.filterwarnings('ignore')
import pandas as pd

import pandas as pd
import numpy as np
from itertools import permutations
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from nltk.stem import PorterStemmer
from sklearn.metrics import classification_report

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer

from nltk.corpus import stopwords  
from nltk.tokenize import word_tokenize  
from nltk.tokenize.treebank import TreebankWordDetokenizer
import csv 
from sklearn.naive_bayes import MultinomialNB

from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfTransformer

from sklearn.metrics import classification_report, accuracy_score

In [2]:
# Let's import and prep the datasets

train=pd.read_csv('not_clean_train_four.csv', sep=',', encoding='utf-8')
test=pd.read_csv('not_clean_test_four.csv', sep=',', encoding='utf-8')

train.info()
train.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4698 entries, 0 to 4697
Data columns (total 3 columns):
text      4698 non-null object
label     4698 non-null object
target    4698 non-null int64
dtypes: int64(1), object(2)
memory usage: 110.2+ KB


,text,label,target
0,শালা মাতারচুদ খানকির পোলা ফলো করার মানুষ পাইলিনা,Personal,0
1,"আবে ছালে রানু মন্ডল তো মেন্টাল, একজন পাগল এর ...",Personal,0
2,ভারতীয় আধিপত্যবাদের বিরুদ্ধে সোচ্চার হতে দেখা ...,Geopolitical,3
3,ফেমিলি সো শালার তোর ভিডিওর শুরুতে আহ সাউন্ড।,Personal,0
4,স্কুল ছুটির পর গেট দিয়ে বের হওয়ার সময পিছন থেক...,Personal,0


In [3]:
test.info()
test.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
text      1000 non-null object
label     1000 non-null object
target    1000 non-null int64
dtypes: int64(1), object(2)
memory usage: 23.5+ KB


,text,label,target
0,বৌদির দুধ দেকে তো আমার ই চোখ ঠিক ছিলো না - পোল...,Personal,0
1,এই সরকার কে যারা নির্লজ্জের মত সাপোর্ট দিয়েছে ...,Political,1
2,পিলখানা হত্যাকান্ড বাংলাদেশের প্রতিরক্ষা ব্যবস...,Geopolitical,3
3,ভারতের অর্থনীতি নিয়ে আপনাদের ভাবতে হবে না। ভা...,Geopolitical,3
4,খানকির পুলা মালায়নদের মেরে সাফা করে ফেল,Personal,0


In [4]:
stop_words='stopwords-bn.txt'
text_data=[]

with open(stop_words,'r',encoding='utf-8') as temp_output_file:
    reader=csv.reader(temp_output_file, delimiter='\n')
    for row in reader:
        text_data.append(row)
stop_word_list=[x[0] for x in text_data]

In [5]:
stop_words = set(stop_word_list)  

def textCleaner(example_sent): 
    word_tokens = word_tokenize(example_sent)  
    #filtered_sentence = [w for w in word_tokens if not w in stop_words]
    filtered_train = TreebankWordDetokenizer().detokenize(word_tokens)

    return filtered_train        

In [6]:
filtered_test = test['text'].apply(textCleaner)

In [7]:
filtered_train = train['text'].apply(textCleaner)

In [8]:
train['label'].value_counts()

Personal        1798
Geopolitical    1457
Religious        784
Political        659
Name: label, dtype: int64

In [9]:
train['target'].value_counts()

0    1798
3    1457
2     784
1     659
Name: target, dtype: int64

In [10]:
x_train, y_train = filtered_train, train['target'].values
x_test, y_test = filtered_test, test['target'].values

In [11]:
#Initialize the `tfidf_vectorizer` 

tfidf_vectorizer = CountVectorizer()
#Fit and transform the training data 
x_train = tfidf_vectorizer.fit_transform(x_train)
#Transform the test set 
x_test = tfidf_vectorizer.transform(x_test)

In [12]:
tfidf_vectorizer.vocabulary_

{'রচ': 1791,
 'নক': 1275,
 'ফল': 1445,
 'কর': 714,
 'ইল': 325,
 'আব': 198,
 'মন': 1652,
 'ডল': 1089,
 'একজন': 477,
 'গল': 848,
 'এর': 551,
 'আচরণ': 150,
 'এমন': 542,
 'হওয়': 2147,
 'কথ': 683,
 'অব': 65,
 'আছ': 152,
 'রত': 1810,
 'আধ': 172,
 'পত': 1368,
 'যব': 1744,
 'হত': 2160,
 'রক': 1776,
 'আসল': 243,
 'লপ': 1925,
 'ওর': 614,
 'আহ': 245,
 'উন': 385,
 'পর': 1390,
 'সময': 2093,
 'ছন': 938,
 'বল': 1542,
 'রদ': 1822,
 'আয়': 253,
 'কত': 673,
 'খর': 785,
 'এখ': 504,
 'এস': 564,
 'গর': 842,
 'সন': 2058,
 'করত': 720,
 'এই': 446,
 'আম': 204,
 'সল': 2111,
 'নদ': 1298,
 'জন': 983,
 'সমব': 2090,
 'দন': 1191,
 'আচ': 148,
 'রশ': 1871,
 'করছ': 717,
 'উপর': 402,
 'অত': 30,
 'রই': 1770,
 'অর': 90,
 'আওয়': 134,
 'হল': 2182,
 'নড়': 1342,
 'আমর': 208,
 'এন': 532,
 'সকল': 2037,
 'অভ': 81,
 'আন': 174,
 'এখন': 506,
 'এট': 515,
 'য়দ': 2230,
 'ইন': 299,
 'আর': 212,
 'কয়': 750,
 'কট': 668,
 'এব': 538,
 'এগ': 510,
 'এরপর': 556,
 'ধর': 1251,
 'আযহ': 211,
 'ইত': 294,
 'কখন': 656,
 'চল': 916,
 'গত': 819,
 'এক': 4

# Naive Bayes

In [12]:
# Let's run the model and get its accuracy score

nb = MultinomialNB(alpha = 0.1)
nb.fit(x_train, y_train)
y_pred = nb.predict(x_test)

categories = ['Personal', 'Geopolitical','Religious','Political']
print(classification_report(y_test, y_pred, target_names=categories))

from sklearn.metrics import matthews_corrcoef
print('MCC:', matthews_corrcoef(y_test, y_pred))

              precision    recall  f1-score   support

    Personal       0.70      0.75      0.72       391
Geopolitical       0.59      0.37      0.45       155
   Religious       0.64      0.56      0.60       173
   Political       0.61      0.73      0.67       281

    accuracy                           0.65      1000
   macro avg       0.64      0.60      0.61      1000
weighted avg       0.65      0.65      0.64      1000

MCC: 0.5069699279225371


# KNN

In [18]:
# Let's run the model and get its accuracy score

knn = KNeighborsClassifier(n_neighbors=12)
knn.fit(x_train, y_train)
y_pred = knn.predict(x_test)

categories = ['Personal', 'Geopolitical','Religious','Political']
print(classification_report(y_test, y_pred, target_names=categories))

#from sklearn.metrics import matthews_corrcoef
print('MCC:', matthews_corrcoef(y_test, y_pred))

NB: 
    MCC: 0.5069699279225371
    P: 0.65
    R: 0.65
    F1: 0.64
KNN: 
    MCC: 0.38146812800251834
    P: 0.6
    R: 0.56
    f1: 0.5
        
SVM: 
    MCC: 0.5327963238498397
    P: 0.67
    R: 0.67
    F1: 0.66
LR: 
    MCC: 0.5417725759571228
    P: 0.68
    R: 0.68
    F1: 0.67
RF: 
    MCC: 0.5599981362428225
    P: 0.69
    R: 0.69
    f1: 0.68
        
GBT: 
    MCC: 0.56801039026076
    P: 0.71
    R: 0.69
    f1: 0.68

              precision    recall  f1-score   support

    Personal       0.49      0.95      0.65       391
Geopolitical       0.38      0.03      0.06       155
   Religious       0.67      0.19      0.30       173
   Political       0.82      0.54      0.65       281

    accuracy                           0.56      1000
   macro avg       0.59      0.43      0.41      1000
weighted avg       0.60      0.56      0.50      1000

MCC: 0.38146812800251834


# SVM

In [19]:
# Let's run the model and get its accuracy score
from sklearn.svm import SVC
svm=SVC(kernel="linear", degree=5, probability=True) # 0.9306853582554517

svm.fit(x_train, y_train)
y_pred = svm.predict(x_test)

categories = ['Personal', 'Geopolitical','Religious','Political']
print(classification_report(y_test, y_pred, target_names=categories))

print('MCC:', matthews_corrcoef(y_test, y_pred))

              precision    recall  f1-score   support

    Personal       0.65      0.87      0.74       391
Geopolitical       0.57      0.33      0.42       155
   Religious       0.70      0.54      0.61       173
   Political       0.74      0.67      0.71       281

    accuracy                           0.67      1000
   macro avg       0.67      0.60      0.62      1000
weighted avg       0.67      0.67      0.66      1000

MCC: 0.5327963238498397


# Gaussian NB

In [21]:
gnb = GaussianNB()
gnb.fit(x_train.toarray(), y_train)

y_pred = gnb.predict(x_test.toarray())

categories = ['Personal', 'Geopolitical','Religious','Political']
print(classification_report(y_test, y_pred, target_names=categories))

print('MCC:', matthews_corrcoef(y_test, y_pred))

              precision    recall  f1-score   support

    Personal       0.50      0.15      0.23       391
Geopolitical       0.23      0.71      0.35       155
   Religious       0.28      0.51      0.36       173
   Political       0.47      0.17      0.25       281

    accuracy                           0.30      1000
   macro avg       0.37      0.38      0.30      1000
weighted avg       0.41      0.30      0.28      1000

MCC: 0.14996247487606365


# LR

In [22]:
# Let's run the model and get its accuracy score

lr = LogisticRegression()
lr.fit(x_train, y_train)

y_pred = lr.predict(x_test)

categories = ['Personal', 'Geopolitical','Religious','Political']
print(classification_report(y_test, y_pred, target_names=categories))

print('MCC:', matthews_corrcoef(y_test, y_pred))

              precision    recall  f1-score   support

    Personal       0.66      0.84      0.74       391
Geopolitical       0.61      0.41      0.49       155
   Religious       0.67      0.55      0.60       173
   Political       0.75      0.68      0.72       281

    accuracy                           0.68      1000
   macro avg       0.67      0.62      0.64      1000
weighted avg       0.68      0.68      0.67      1000

MCC: 0.5417725759571228


# Decision Tree Classifier

In [23]:
dt = DecisionTreeClassifier()

dt.fit(x_train, y_train)
y_pred = dt.predict(x_test)

categories = ['Personal', 'Geopolitical','Religious','Political']
print(classification_report(y_test, y_pred, target_names=categories))

print('MCC:', matthews_corrcoef(y_test, y_pred))

              precision    recall  f1-score   support

    Personal       0.67      0.66      0.67       391
Geopolitical       0.43      0.41      0.42       155
   Religious       0.49      0.55      0.52       173
   Political       0.69      0.68      0.68       281

    accuracy                           0.61      1000
   macro avg       0.57      0.57      0.57      1000
weighted avg       0.61      0.61      0.61      1000

MCC: 0.45026874887830265


# Random Forest

In [24]:
# Let's run the model and get its accuracy score

rf = RandomForestClassifier()
rf.fit(x_train, y_train)
y_pred = rf.predict(x_test)

categories = ['Personal', 'Geopolitical','Religious','Political']
print(classification_report(y_test, y_pred, target_names=categories))

print('MCC:', matthews_corrcoef(y_test, y_pred))

              precision    recall  f1-score   support

    Personal       0.68      0.84      0.75       391
Geopolitical       0.65      0.34      0.45       155
   Religious       0.69      0.60      0.64       173
   Political       0.73      0.73      0.73       281

    accuracy                           0.69      1000
   macro avg       0.69      0.63      0.64      1000
weighted avg       0.69      0.69      0.68      1000

MCC: 0.5599981362428225


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from nltk.stem import PorterStemmer
from sklearn.metrics import classification_report

# Gradient Boosted Trees

In [25]:
# Let's run the model and get its accuracy score

gbt = GradientBoostingClassifier(random_state=0, n_estimators=200)
gbt.fit(x_train, y_train)
y_pred = gbt.predict(x_test)

categories = ['Personal', 'Geopolitical','Religious','Political']
print(classification_report(y_test, y_pred, target_names=categories))

print('MCC:', matthews_corrcoef(y_test, y_pred))

              precision    recall  f1-score   support

    Personal       0.63      0.90      0.74       391
Geopolitical       0.72      0.35      0.47       155
   Religious       0.74      0.57      0.64       173
   Political       0.81      0.68      0.74       281

    accuracy                           0.69      1000
   macro avg       0.72      0.62      0.65      1000
weighted avg       0.71      0.69      0.68      1000

MCC: 0.56801039026076


In [90]:
test.target.unique()

array([0, 1, 3, 2])

In [123]:
import eli5
eli5.show_weights(gbt, vec=tfidf_vectorizer, top=15, target_names = test.target.unique())

Weight,Feature
0.3807 ± 0.3164,রত
0.0772 ± 0.1494,উন
0.0399 ± 0.1186,আওয়
0.0243 ± 0.0916,ইসল
0.0221 ± 0.0844,ধর
0.0174 ± 0.0890,নক
0.0167 ± 0.0904,আল
0.0156 ± 0.0904,আওয
0.0148 ± 0.0615,রতক
0.0129 ± 0.0848,সল


# AdaBoostClassifier

In [21]:
# Let's run the model and get its accuracy score

ab = AdaBoostClassifier(random_state=0, n_estimators=200)
ab.fit(x_train, y_train)
y_pred = ab.predict(x_test)

categories = ['Personal', 'Geopolitical','Religious','Political']
print(classification_report(y_test, y_pred, target_names=categories))

print('MCC:', matthews_corrcoef(y_test, y_pred))

              precision    recall  f1-score   support

    Personal       0.66      0.88      0.75       391
Geopolitical       0.58      0.39      0.47       155
   Religious       0.64      0.54      0.59       173
   Political       0.80      0.66      0.72       281

    accuracy                           0.68      1000
   macro avg       0.67      0.62      0.63      1000
weighted avg       0.68      0.68      0.67      1000



# LGBMClassifier

In [ ]:
# Let's run the model and get its accuracy score

LGBM = LGBMClassifier(random_state=0, n_estimators=200)
LGBM.fit(x_train, y_train)
y_pred = LGBM.predict(x_test)

categories = ['Personal', 'Geopolitical','Religious','Political']
print(classification_report(y_test, y_pred, target_names=categories))

# PassiveAggressiveClassifier

In [52]:
from sklearn.linear_model import PassiveAggressiveClassifier

PAC = PassiveAggressiveClassifier(max_iter=10000, random_state=0, tol=1e-5)

PAC.fit(x_train, y_train)
y_pred = PAC.predict(x_test)

categories = ['Personal', 'Geopolitical','Religious','Political']
print(classification_report(y_test, y_pred, target_names=categories))

              precision    recall  f1-score   support

    Personal       0.66      0.74      0.70       391
Geopolitical       0.58      0.40      0.47       155
   Religious       0.64      0.55      0.59       173
   Political       0.66      0.71      0.68       281

    accuracy                           0.65      1000
   macro avg       0.63      0.60      0.61      1000
weighted avg       0.64      0.65      0.64      1000



In [54]:
from sklearn.feature_selection import SelectFromModel

import matplotlib.pyplot as plt
import numpy as np

from sklearn.datasets import load_diabetes
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LassoCV

diabetes = load_diabetes()

X = diabetes.data
y = diabetes.target

In [55]:
feature_names = diabetes.feature_names
print(feature_names)

clf = LassoCV().fit(X, y)
importance = np.abs(clf.coef_)
print(importance)

idx_third = importance.argsort()[-3]
threshold = importance[idx_third] + 0.01

idx_features = (-importance).argsort()[:2]
name_features = np.array(feature_names)[idx_features]
print('Selected features: {}'.format(name_features))

sfm = SelectFromModel(clf, threshold=threshold)
sfm.fit(X, y)
X_transform = sfm.transform(X)

n_features = sfm.transform(X).shape[1]

['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
[  6.49684455 235.99640534 521.73854261 321.06689245 569.4426838
 302.45627915   0.         143.6995665  669.92633112  66.83430445]
Selected features: ['s5' 's1']


In [58]:
importance = np.abs(svm.coef_)

idx_third = importance.argsort()[-3]
threshold = importance[idx_third] + 0.01

sfm = SelectFromModel(svm, threshold=threshold)
sfm.fit(x_train, y_train)
X_transform = sfm.transform(x_train)

n_features = sfm.transform(X).shape[1]

ValueError: WRITEBACKIFCOPY base is read-only